In [1]:
%%writefile /content/trocr_engine.py
import cv2
import numpy as np
from PIL import Image
import fitz
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

_processor = None
_model = None


def _load_model():
    global _processor, _model
    if _model is None:
        model_name = "microsoft/trocr-base-printed"
        _processor = TrOCRProcessor.from_pretrained(model_name)
        _model = VisionEncoderDecoderModel.from_pretrained(model_name)
    return _processor, _model


def _segment_lines(image_path, min_line_height=15):
    img = cv2.imread(image_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    row_sums = np.sum(binary, axis=1)

    in_line = False
    line_bands = []
    start = 0
    for y, val in enumerate(row_sums):
        if val > 0 and not in_line:
            in_line = True
            start = y
        elif val == 0 and in_line:
            in_line = False
            if y - start >= min_line_height:
                line_bands.append((start, y))
    if in_line:
        line_bands.append((start, len(row_sums)))

    pil_img = Image.open(image_path).convert("RGB")
    width = pil_img.width
    line_crops = []
    padding = 4
    for (y1, y2) in line_bands:
        y1_pad = max(0, y1 - padding)
        y2_pad = min(pil_img.height, y2 + padding)
        crop = pil_img.crop((0, y1_pad, width, y2_pad))
        line_crops.append(crop)

    return line_crops


def _ocr_single_line(line_image):
    processor, model = _load_model()
    pixel_values = processor(images=line_image, return_tensors="pt").pixel_values
    generated_ids = model.generate(pixel_values, max_new_tokens=64)
    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return text


def ocr_image(image_path):
    line_crops = _segment_lines(image_path)

    if not line_crops:
        processor, model = _load_model()
        image = Image.open(image_path).convert("RGB")
        pixel_values = processor(images=image, return_tensors="pt").pixel_values
        generated_ids = model.generate(pixel_values, max_new_tokens=64)
        return processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    print(f"[INFO] Segmented into {len(line_crops)} line(s)")
    lines_text = []
    for i, line_img in enumerate(line_crops):
        line_text = _ocr_single_line(line_img)
        print(f"[INFO]   line {i+1}: {line_text}")
        lines_text.append(line_text)

    return "\n".join(lines_text)


def ocr_pdf_as_images(pdf_path, dpi=200):
    doc = fitz.open(pdf_path)
    all_text = []
    zoom = dpi / 72
    mat = fitz.Matrix(zoom, zoom)

    for page_num, page in enumerate(doc):
        pix = page.get_pixmap(matrix=mat)
        img_path = f"/tmp/_page_{page_num}.png"
        pix.save(img_path)
        page_text = ocr_image(img_path)
        all_text.append(page_text)

    doc.close()
    return "\n".join(all_text)


if __name__ == "__main__":
    import sys
    if len(sys.argv) < 2:
        print("Usage: python trocr_engine.py path/to/image_or_scanned.pdf")
        sys.exit(1)

    path = sys.argv[1]
    if path.lower().endswith(".pdf"):
        result = ocr_pdf_as_images(path)
    else:
        result = ocr_image(path)

    print(result)

Overwriting /content/trocr_engine.py


In [2]:
!grep -c "_segment_lines" /content/trocr_engine.py
!wc -l /content/trocr_engine.py

2
110 /content/trocr_engine.py


In [1]:
!pip install transformers==4.46.3 sentencepiece pymupdf torch pillow jiwer opencv-python-headless -q

In [2]:
!python run_ocr.py 'Screenshot 2026-09-08 072349.png'

2026-09-08 04:55:48.500678: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
[INFO] Detected: image file -> using TrOCR
[INFO] Segmented into 23 line(s)
Config of the encoder: <class 'transformers.models.vit.modeling_vit.ViTModel'> is overwritten by shared encoder config: ViTConfig {
  "attention_probs_dropout_prob": 0.0,
  "encoder_stride": 16,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "image_size": 384,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_norm_eps": 1e-12,
  "model_type": "vit",
  "num_attention_heads": 12,
  "num_channels": 3,
  "num_hidden_layers": 12,
  "patch_size": 16,
  "qkv_bias": false,
  "transformers_version": "4.46.3"
}

Config of the decoder: <class 'transformers.mode

In [3]:
!pip install easyocr -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 30.4 MB/s eta 0:00:00


In [4]:
%%writefile /content/trocr_engine.py
import easyocr
import fitz

_reader = None


def _load_reader():
    global _reader
    if _reader is None:
        print("[INFO] Loading Arabic OCR model (first time only, may take a minute)...")
        _reader = easyocr.Reader(['ar', 'en'])
    return _reader


def ocr_image(image_path):
    reader = _load_reader()
    results = reader.readtext(image_path, detail=0, paragraph=True)
    return "\n".join(results)


def ocr_pdf_as_images(pdf_path, dpi=200):
    doc = fitz.open(pdf_path)
    all_text = []
    zoom = dpi / 72
    mat = fitz.Matrix(zoom, zoom)

    for page_num, page in enumerate(doc):
        pix = page.get_pixmap(matrix=mat)
        img_path = f"/tmp/_page_{page_num}.png"
        pix.save(img_path)
        page_text = ocr_image(img_path)
        all_text.append(page_text)

    doc.close()
    return "\n".join(all_text)


if __name__ == "__main__":
    import sys
    if len(sys.argv) < 2:
        print("Usage: python trocr_engine.py path/to/image_or_scanned.pdf")
        sys.exit(1)

    path = sys.argv[1]
    if path.lower().endswith(".pdf"):
        result = ocr_pdf_as_images(path)
    else:
        result = ocr_image(path)

    print(result)

Overwriting /content/trocr_engine.py


In [5]:
!python run_ocr.py 'Screenshot 2026-09-08 072349.png'

[INFO] Detected: image file -> using TrOCR
[INFO] Loading Arabic OCR model (first time only, may take a minute)...
Progress: |██████████████████████████████████████████████████| 100.0% CompleteDownloading recognition model, please wait. This may take several minutes depending upon your network connection.
Progress: |██████████████████████████████████████████████████| 100.0% Complete[INFO] Raw text length: 1280 characters
[INFO] Cleaned text length: 1277 characters

--- FINAL OUTPUT ---

باطرض طاظطاء ولرع من لموذلك بلي دار٥ا . لبطران مورئة مع زثار من للدبا . للمور لنادم موم من ئرن نب وزماق ولداعان من الطبد لما مونة اللمرر لدلمل مدذ من لوب ممعر مامرة دبازذ بب للنل لرنب الغرم لم مع نرف رماع لمالوامرن لنرجا لم مورثذ ومطرونا . لتاء معوز
تلغ مساحة مذالثاء ٨٦1ا ماتال غاذبعار لمقظم تربخ ٠ ٨ا٨اا0 -يردصف لبرك 8 بعرن لبارك 8 من لاء مزل من ندقا لولق : دم وول . ونلي انللذ تني مر عترن لابحارك لعالي . م ناء من لالرن تلا لل تبة - طا الادب رلموم الاسنا -
الطلذ الادض ملزن من مدل دني ندن وحم ممنوم . مزم 

In [6]:
# Save results to files so you have them as proof
with open("/content/result_digital_test.txt", "w", encoding="utf-8") as f:
    f.write("Test: PAYMENT screenshot (English, clean image)\nOutput: PAYMENT\n")

with open("/content/result_arabic_test.txt", "w", encoding="utf-8") as f:
    f.write("Test: Arabic legal document (photographed)\nOutput:\n[paste your Arabic output here]\n")

In [7]:
!ls /content/

 cleaning.py			        result_arabic_test.txt
 document_detector.py		        result_digital_test.txt
 evaluate_ocr.py		        run_ocr.py
'NTI_Graduation project_document.pdf'   sample_data
 pdf_extractor.py		       'Screenshot 2026-08-30 185417.png'
 __pycache__			       'Screenshot 2026-09-08 072349.png'
 README.md			        trocr_engine.py
 requirements.txt


In [8]:
!python run_ocr.py 'NTI_Graduation project_document.pdf'

[INFO] Detected: digital PDF -> using PyMuPDF direct extraction
[INFO] Raw text length: 18264 characters
[INFO] Cleaned text length: 14449 characters

--- FINAL OUTPUT ---

NTI_Graduation project
OCR + : مشروع قوي فع ال ، واالهم ان كل تقنية فيه لها دور حقيقيSmart Citizen Assistant المشروع اللي اخترتيه
.Transformers + RAG + Vector DB + Hybrid Search + Router + LLM Chains + Verification
.سيناريو كامل من لحظة ما المواطن يكتب السؤال لحد ما ياخد االجابة خليني اف كهولك من الصفر، وبعدها امشي معاكي في
او ال : المشروع ببساطة بيعمل ايه؟🧠
: مواقع حكومية مختلفة3 تخيلي ان المواطن بدل ما يدخل علي
. واحد ويقول له اي حاجةChatbot يدخل علي
:مث
""عايز اطلع بطاقة رقم قومي الول مرة، ايه االوراق المطلوبة؟
:او
""عندي مخالفة مرورية، اعرف اعمل ايه؟
:/صورة ويقولPDF او يرفع
""المستند ده ناقصه ايه؟
:النظام نفسه يفهم
السؤال تابع النهي جهة؟
:ثم
هل محتاج نبحث في المستندات؟
:ثم
ايه المعلومات الموجودة في القوانين واللوائح المتعلقة بالسؤال؟
:ثم
االحوال المدنية
الضرائب
المرور
.يولد اجابة بنا ء علي المصادر الرسمية
:ثم
.ي

In [13]:
text = r"""NTI_Graduation project
OCR + : مشروع قوي فع ال ، واالهم ان كل تقنية فيه لها دور حقيقيSmart Citizen Assistant المشروع اللي اخترتيه
.Transformers + RAG + Vector DB + Hybrid Search + Router + LLM Chains + Verification
.سيناريو كامل من لحظة ما المواطن يكتب السؤال لحد ما ياخد االجابة خليني اف كهولك من الصفر، وبعدها امشي معاكي في
او ال : المشروع ببساطة بيعمل ايه؟🧠
: مواقع حكومية مختلفة3 تخيلي ان المواطن بدل ما يدخل علي
. واحد ويقول له اي حاجةChatbot يدخل علي
:مث
""عايز اطلع بطاقة رقم قومي الول مرة، ايه االوراق المطلوبة؟
:او
""عندي مخالفة مرورية، اعرف اعمل ايه؟
:/صورة ويقولPDF او يرفع
""المستند ده ناقصه ايه؟
:النظام نفسه يفهم
السؤال تابع النهي جهة؟
:ثم
هل محتاج نبحث في المستندات؟
:ثم
ايه المعلومات الموجودة في القوانين واللوائح المتعلقة بالسؤال؟
:ثم
االحوال المدنية
الضرائب
المرور
.يولد اجابة بنا ء علي المصادر الرسمية
:ثم
.يتحقق ان االجابة فع ال مدعومة بالمصادر
.Chatbot يعني مش مجرد
:هو اقرب الي
Intelligent Government Information System
الصورة الكبيرة للمشروع🏗️
: طبقات7 المشروع عندك ممكن نفهمه كـ
👤 Citizen
│
▼
┌───────────────┐
│ User Input │
│ Text / PDF / │
│ Image │
└───────┬───────┘
│
┌────────┴────────┐
│ │
▼ ▼
Text Input Document
│
▼
OCR
│
▼
Extracted Text
│
└──────┐
▼
┌─────────────┐
│ Router LLM │
└──────┬──────┘
│
┌──────────────────┼──────────────────┐
▼ ▼ ▼
.End-to-End Pipeline الـ وده هو
1️⃣ User Input
.اول حاجة المواطن بيتعامل مع الواجهة
:ممكن يعمل حاجتين
الحالة االولي: سؤال نصي
:مث
ازاي اطلع شهادة ميالد؟
:هنا
الحالة الثانية: يرفع مستند
:مثال يرفع
Civil Affairs Tax Traffic
│ │ │
▼ ▼ ▼
RAG DB RAG DB RAG DB
│ │ │
└──────────────────┼──────────────────┘
▼
Generation LLM
│
▼
Verification LLM
│
▼
Final Answer
User Question
↓
Router
PDF
:او
:ويقول
""المستند ده خاص بايه؟
: اضافيPipeline هنا عندنا
2️⃣ OCR 📄 → Text
.دي مرحلة مهمة ج دا في مشروعك
. مش هيفهم الصورة الخام بالطريقة اللي احنا عايزينها هناLLM الن الـ
:لو المواطن رفع صورة
: يحولها اليOCR الـ
Image
Document
↓
Document Processing
↓
OCR
↓
Extracted Text
↓
Router
┌─────────────────────┐
│ رسوم سداد ايصال │
│ مصلحة ... │
│ جنيه 75 :المبلغ │
│ التاريخ: ... │
└─────────────────────┘
"رسوم سداد ايصال
مصلحة ...
جنيه 75 المبلغ
التاريخ ..."
:وهنا عندك حالتين
PDF فيه Text اص
:نستخدم
PyMuPDF
.ونستخرج النص مباشرة
. الن النص اصال موجودOCR وده افضل من
PDF عبارة عن Scan / Image
.text مش هيقدر يقرا الكالم كـPyMuPDF هنا
:فنستخدم
TrOCR
. في المشروعTransformer وده جزء جميل ج دا الظهار استخدام
بالذات؟TrOCR ليه3️⃣
:الن
TrOCR = Transformer-based OCR
. مستخدم في معالجة المستنداتTransformer فان ت هنا عندك
PDF
↓
PyMuPDF
↓
Text
Image
↓
TrOCR
↓
Text
.embeddings تاني للـTransformer وبعدين ممكن يكون عندك
.، وده ممتاز للمناقشةTransformer يعني في المشروع عندك اكتر من استخدام للـ
Cleaning → بعد استخراج النص4️⃣
.%100 غال با مش هيكون نظيفOCR النص اللي خارج من
:مث
:نحتاج نعمل
:فيطلع
5️⃣ Chunking
.RAG دي نقطة مهمة ج دا في الـ
:افترضي ان عندك قانون كامل
. كل مرةprompt مش منطقي تحطي القانون كله في الـ
:فنقسمه الي اجزاء
رسوم سداد ايصال!!!
المدنية الحوال مصلحة
Cleaning
↓
Remove unnecessary symbols
↓
Normalize spaces
↓
Normalize Arabic text
رسوم سداد ايصال
المدنية االحوال مصلحة
100 صفحة
:مث
. واستكملت في اللي بعده، مانفقدش السياقChunk مهم عشان لو معلومة بدات في نهايةoverlap الـ
Knowledge Base بناء6️⃣
.هنا بنجهز المعلومات الحكومية
:مثال لالحوال المدنية
:وللضرائب
Document
│
├── Chunk 1
├── Chunk 2
├── Chunk 3
├── Chunk 4
├── ...
└── Chunk 100
Chunk size = 300 tokens
Overlap = 50 tokens
Civil Affairs
│
├── قوانين
├── لوائح
├── المستندات استخراج شروط
├── المطلوبة االوراق
├── الرسوم
├── االجراءات
└── FAQ
Tax
│
├── الضرائب قوانين
├── اللوائح
├── االجراءات
├── النماذج
└── FAQ
:وللمرور
7️⃣ Embeddings 🧠
.Vector الزم يتحول اليChunk دلوقتي كل
:مث
.Transformer embedding model يدخل الي
:مث
:فينتج
. رياضيrepresentation بقي لهChunk وبالتالي كل
8️⃣ Vector Database
: دي فيvectors نخزن الـ
:او
Traffic
│
├── المرور قوانين
├── المخالفات
├── االجراءات
├── التراخيص
└── FAQ
"بطاقة الستخراج المطلوبة االوراق..."
multilingual-e5
[0.12, -0.31, 0.77, 0.05, ...]
FAISS
:وممكن تعملي
. منفصلةKnowledge Base وده بيدعم فكرة ان كل جهة لها
🔎 ؟Hybrid Search طب ليه9️⃣
.دي نقطة قوية ج دا في المشروع
:Search عندك نوعين
Dense Search
.بيعتمد علي المعني
:مث
""عايز اعمل بطاقة
:والمستند مكتوب
""اجراءات استخراج بطاقة تحقيق الشخصية
. ممكن يفهم انهم قريبين في المعنيembedding رغم ان الكلمات مختلفة، الـ
BM25
.بيعتمد علي الكلمات نفسها
:مثال المواطن قال
"7 "نموذج
: ممتاز في العثور علي المستند اللي يحتوي عليBM25 فـ
Chroma
Civil Affairs → Vector DB
Tax → Vector DB
Traffic → Vector DB
Hybrid Search
:نجمع الثنين
. اقوي من استخدام نوع واحد فقطRetrieval وده يخلي الـ
🚦 Router : دلوقتي ناتي الهم جزء🔟
.Intelligent هنا بقي المشروع يبدا يبقي
: يستقبلRouter الـ
:ويقرر
:مث
""ازاي اطلع بطاقة شخصية؟
:يرجع
7 نموذج
Query
│
┌────────┴────────┐
▼ ▼
Dense Search BM25
│ │
└────────┬────────┘
▼
Ranked Results
Question
+
OCR Text (if available)
Which department?
""عايز اعرف الضريبة المطلوبة علي العقار
:يرجع
""عندي مخالفة مرور
:يرجع
؟LLM دهRouter هل الـ🧠
.ممكن يكون عندك اختيارين
Option 1 — LLM Router
LLM ياخد السؤال ويرجع:
Option 2 — Transformer Classifier
: لـfine-tuning تعملي
AraBERT
: ان ت جهزتيهdataset علي
civil_affairs
tax
traffic
{
"department": "traffic"
}
Question Label
. كمانTransformer fine-tuning وده يخلي عندك
Router بعد الـ1️⃣1️⃣
:نفترض السؤال
""ايه االوراق المطلوبة الستخراج بطاقة رقم قومي؟
Router قال:
:اذن
:مش هنبحث في
.وده مهم ج دا
1️⃣2️⃣ RAG Pipeline
: دلوقتي ياخذ السؤالRAG الـ
بطاقة؟ اطلع ازاي Civil
البطاقة؟ استخراج اوراق هي ما Civil
الضريبة؟ ادفع ازاي Tax
العقار؟ ضريبة هي ما Tax
مرور مخالفة عندي Traffic
الرخصة؟ اجدد ازاي Traffic
civil_affairs
User
↓
Router
↓
Civil Affairs RAG
Traffic
Tax
:يعمل
:وفي نفس الوقت
:وبعدين
:مثال يرجع
1️⃣3️⃣ Generation LLM ✨
.يكتب اجابة الول مرة يبداLLM دلوقتي الـ
: ممكن يكون بشكل منطقيPrompt الـ
قومي؟ رقم بطاقة الستخراج المطلوبة االوراق ايه
Query
↓
Embedding
↓
Dense Search
Query
↓
BM25
Dense Results
+
BM25 Results
↓
Hybrid Ranking
↓
Top 5 Chunks
Chunk 17
Chunk 42
Chunk 63
Chunk 81
Chunk 92
: يولدLLM والـ
"... "الستخراج بطاقة الرقم القومي، يلزم تقديم
:واالهم
1️⃣4️⃣ Verification LLM 🛡️
.دي من اقوي اجزاء المشروع
:بدل ما نقول
LLM قال االجابة اذن خالص.
.Verifier تاني كـLLM نعمل
:ياخد
:ويسال
You are a government information assistant.
Answer the citizen's question using ONLY
the provided official context.
Question:
...
Retrieved Context:
...
If the answer is not supported by the context,
say that the information is unavailable.
Sources:
[Document 17]
[Document 42]
Generated Answer
+
Retrieved Context
هل االجابة مدعومة فعال بالمصادر؟
:مث
Verifier يكتشف:
.فيمنع االجابة او يطلب تصحيحها
كاملةChain وبالتالي عندك🔥
:السؤال الواحد ممكن يمشي
Answer:
شخصية صور 3 تقديم يجب.
Source:
شخصيتين صورتين تقديم يجب.
CONTRADICTION
User
↓
Document?
/ \
Yes No
↓ ↓
OCR Text
\ /
↓ ↓
Router
↓
┌─────────┼─────────┐
↓ ↓ ↓
Civil Tax Traffic
↓ ↓ ↓
RAG RAG RAG
└─────────┼─────────┘
↓
Retrieved Docs
↓
Generation LLM
↓
.LLM Chain وده بالضبط معني ان عندك
؟Query Rewriting طيب فين الـ🚨
.دي اضافة انا انصحك ج دا تحطيها
:مثال المواطن يقول
""ازاي اطلع بطاقة؟
.النظام يجاوب
:بعدها يقول
""طب لو البطاقة ضاعت؟
.ambiguous السؤال الثاني لوحده
:conversation يفهم الـLLM لكن الـ
:ويعيد صياغتها
:ثم
Draft Answer
↓
Verification LLM
↓
Final Answer
↓
Citizen
Previous:
بطاقة استخراج اجراءات
Current:
ضاعت؟ البطاقة لو طب
القومي؟ الرقم بطاقة فقدان حالة في المطلوبة االجراءات ما
Query Rewriting
↓
.conversational  فعchatbot دي هتخلي الـ🔥
ونرجع للمستند📄
:نفترض المواطن رفع صورة وقال
""المستند ده خاص بايه؟
:الصورة
: يقولRouter لكن هنا ممكن
:وبعدين
:فيقول
"... "المستند يبدو كايصال سداد رسوم، ويشير الي
:ولو المواطن يسال
RAG
↓
Generation
Government Document
↓
OCR
↓
"رسوم سداد ايصال..."
↓
Router
document_analysis
OCR Text
+
User Question
↓
Relevant RAG
↓
LLM
""هل المستند ده كافي لتقديم الطلب؟
:هنا الموضوع اقوي
:مث
:ويقول
"..."وف قا للمستندات المسترجعة، يبدو ان هناك مستن دا اضاف يا مطلو با
.من غير ما يدعي يقين غير موجود
؟LangChain / LangGraph طب فين الـ🧩
.هنا هما بيساعدوكي تربطي كل االجزاء
:workflow ممكن يمثل الـLangGraph مث
OCR
↓
Identify Document
↓
Retrieve Requirements
↓
Compare
↓
LLM
↓
Missing Documents
✓ شخصية صورة
✓ السداد ايصال
✗ القومي الرقم بطاقة صورة
START
↓
Document Check
↓
OCR
↓
Router
↓
:conditional edges يعملRouter والـ
.conditional routing وbranches فيهworkflow الن الـLangGraph وده استخدام ممتاز ج دا لـ
Evaluation واخي را🧪
:مش هينفع نقول
"."المشروع شغال وخالص
.الزم تثبتوا باالرقام انه شغال
Router
:مث
Department
↓
Query Rewrite
↓
Hybrid Retrieval
↓
Generation
↓
Verification
↓
END
Router
│
┌──────────┼──────────┐
↓ ↓ ↓
Civil Tax Traffic
│ │ │
RAG RAG RAG
Accuracy
Precision
Recall
F1-score
Retrieval
:نقيس
:مث
.relevant information نتائج، النظام غال با بيجيب الـ5 يعني من ضمن افضل
RAG
:ممكن تستخدموا
:وتقيسوا حاجات مثل
واهم نقطة في المشروع كله🏆
.Chatbot ما تقدمهوش علي انه انا شايف انكم
:قدموه علي انه
Router Accuracy = 94%
Recall@K
Precision@K
MRR
Recall@5 = 91%
RAGAS
Faithfulness
Answer Relevancy
Context Precision
Context Recall
An Intelligent Multi-Department Government Information System powered by OCR,
Transformers, Hybrid RAG, LLM Routing, and Multi-Stage Verification.
. تقلل من قيمة اللي عملتوهChatbot الن كلمة
:المشروع فعل يا عبارة عن
ولو هتشتغلوا عليه كفريق🎯
:Modules 6 انا اقترح تقسيمه الي
. واحدEnd-to-End system وبكده كل عضو في الفريق عنده جزء حقيقي، وفي االخر كل االجزاء بتتجمع في
SMART CITIZEN ASSISTANT
│
┌─────────────┴─────────────┐
│ │
Document Intelligence Conversational AI
│ │
OCR Router
│ │
TrOCR Department
│ │
└─────────────┬─────────────┘
│
Hybrid RAG
│
Transformer
Embeddings
│
Vector DB
│
LLM Chain
│
Verification
│
Final Answer
1. Document/OCR Module → PyMuPDF + TrOCR
2. Data & Knowledge Base Module → scraping/collection + cleaning + chunking
3. Retrieval Module → Embeddings + FAISS/Chroma + BM25 + Hybrid Search
4. Router Module → AraBERT/LLM classification
5. LLM Pipeline Module → LangGraph + RAG + Generation + Verification
6. Evaluation & UI Module → RAGAS + metrics + Streamlit/Gradio
:، ونحدد بالضبطNode-by-Node النهائية للمشروعArchitecture نرسم الـ:والخطوة اللي انصح نعملها بعد كده مباشرة
بتاع input/output والـ ،امتي بيتنادي LLM كل ،فين هنستخدمه Dataset كل ،ايه هنستخدمه Model كل
.ايه؛ الن دي هتبقي الخريطة اللي هتمشوا عليها في التنفيذ والمناقشة Node كل
#BUILDING : لفع ا موجودة المشروع عليها بنينا اللي االساسية الداتا
4. Egyptian Legal Corpus — ⭐⭐⭐⭐⭐
.وده اهم جزء في المشروع تقري با
dataflare/egypt-legal-corpus
Egyptian Legal Corpus — Hugging Face
:والمفاجاة ان حجمه كويس ج دا
. حسب بطاقة البياناتMIT موصوف بانه مستخرج من وثائق قانونية مصرية رسمية، ومرخصdataset والـ
:RAG ده هيكون اساس الـ
2,434 records
token مليون25 حوالي
MB 24.9 حوالي
Arabic
Egyptian legal texts
Structured metadata
Hierarchical legal categories
law_name
text
categories
tokens
Egyptian Legal Corpus
↓
Cleaning
↓
Chunking
↓
Embeddings
↓
FAISS / Chroma
↓
Hybrid Retrieval
.Knowledge Base للـ عنها اتكلمنا اللي من Dataset اقوي عندي وده
5. QA_LAW_Egyptian_dataset — ✅ لموجود فع ا
: اللي كن ت بتتكلمي عنها ب السمDataset لقيت الـ
Omar-youssef/QA_LAW_Egyptian_dataset
QA_LAW_Egyptian_dataset — Hugging Face
:وفيها
.Apache-2.0 المذكورlicense والـ
.Evaluation دي ممتازة للـ
:مث
:لكن فيه تحذير مهم
. نفسهاKnowledge Base ما انصحش نستخدمها كـ
:االفضل
:و
سؤال/اجابة3,725
Egyptian Arabic
unique legal topics 738 حوالي
question
answer
source_topics
Question
↓
Our RAG
↓
Generated Answer
↓
Compare with Reference Answer
Legal Corpus
↓
Knowledge Base
.Data Leakage عشان ما نعملش
⭐⭐⭐⭐⭐ — قانونية مصرية اكبرDataset . لقيت6
:ودي بصراحة اضافة قوية ج دا
tarekys5/egyptian_legal_v2
Egyptian Legal QA v2 — Hugging Face
:حجمها
: فيهاfields واالهم ان الـ
:يعني عندك
السؤال
االساس القانوني
المادة القانونية
االجابة
QA Dataset
↓
Evaluation
9,793 train
516 validation
10,309 اجمالي
Arabic
Egyptian law
instruction
input
legal_basis
output
question_type
article
chapter
: النه عندك عالقة واضحة بينRAG ممتاز ج دا لمشروع وده
. متقدمEvaluation لو هدفناQA_LAW_Egyptian_dataset بصراحة انا افضلها علي
7. Dataset اضافية: Egyptian Legal Multi-Task
:لقيت
fr3on/eg-legal-multi-task
Egyptian Legal Multi-Task — Hugging Face
:، وتجمعexample 1,046 فيها
. واحدةDataset في
: مثلfeature ودي ممكن تكون مفيدة ج دا لو عايزين نضيف
""حدد نوع المستند/المعلومة القانونية
:او
""استخرج الكيانات القانونية
. للمشروعcore dataset ليست الـ لكنها
9. NIYYAH — بديل حديث للـ Intent Classification
:لقيت كمان
Question
↓
Legal Basis
↓
Article
↓
Answer
Classification
QA
NER
Summarization
NIYYAH
NIYYAH Arabic Intent Dataset
:وفيها
10,500 utterances
30 intents
:وفيها
Rout ، لكن مفيدة ج دا لفهم وتصميم الـGovernment برضه مش
طيب هنستخدم ايه فعل يا؟🧠
.هغ ير شوية في الخطة االصلية انا بعد البحث ده
. موجودةDataset مش هنحاول نستخدم كل
: دهData Stack انا اقترح الـ
Saudi Arabic
MSA
Human validation
Train / Validation / Test
Out-of-scope intents
SMART CITIZEN ASSISTANT
│
┌───────────────────┼──────────────────┐
│ │ │
▼ ▼ ▼
OCR Knowledge Base Evaluation
│ │ │
▼ ▼ ▼
Arabic Documents Egyptian Legal QA_LAW_Egyptian
OCR Dataset Corpus + Legal v2
│ │
▼ ▼
TrOCR Embeddings
│
▼
Vector DB
│
النهائي اللي ارشحهDataset Stack الـ🏆
Datasetالحجم
الستخدام
اختياري
Arabic Documents
OCR
10K images
OCR evaluation
⭐⭐⭐⭐⭐
Egypt Legal Corpus
2,434 docs / 25M
tokens
RAG Knowledge Base
⭐⭐⭐⭐⭐
QA_LAW_Egyptian
3,725 QA
RAG evaluation
⭐⭐⭐⭐
Egyptian Legal v2
10,309 QA
Advanced RAG evaluation
⭐⭐⭐⭐⭐
Arabic OCR 2.16M
2.16M
OCR recognition
⭐⭐⭐
Arabic Synthetic
Scans
300K
OCR robustness
⭐⭐⭐⭐
ArBanking77
31K
Router
methodology/baseline
⭐⭐⭐
NIYYAH
10.5K
Intent baseline
⭐⭐⭐
الزم ناخد بالنا منهData Leakage واالهم: عندنا🔥
:لو استخدمنا
.نفس النصوص القانونية، الزم نكون واضحين ج دا في التقرير مبنية منQA Dataset ، وبعدها اختبرنا عليRAG في بناء الـ
: منظمEvaluation االفضل نعمل
▼
Hybrid RAG
Egyptian Legal Corpus
Legal Sources
│
┌──────────┴──────────┐
▼ ▼
Knowledge Base Evaluation Set
│ │
completely ونوضح ده بدل ما نقدمه كـin-domain benchmark ، نعتبرهاcorpus نفسها مشتقة من نفسQA ولو الـ
.unseen test
الخالصة🎯
.ايوه، المشروع قابل للتنفيذ من ناحية الداتا، وبشكل ممتاز
:واقوي تركيبة عندي حال يا هي
OCR:
Arabic Documents OCR Dataset
Knowledge Base:
Egyptian Legal Corpus
Evaluation:
Egyptian Legal v2 + QA_LAW_Egyptian
Router:
Dataset نعملها احنا للـ Civil / Tax / Traffic، مع الستفادة من ArBanking77 كـ baseline للـ intent classification.
OCR robustness:
Arabic Synthetic Scans 300K كاختبار اضافي.
بتاعتها واضحة ومناسبة لالستخدام البحثي؛ لكن بالنسبةlicenses والجميل ان المصادر االساسية كلها متاحة حال يا، وبعضها
المشروعdataset هنحتاج نتحقق من المصدر الرسمي/حقوق اعادة التوزيع قبل ما نحطها في ،للنصوص الحكومية نفسها
.المنشورة، حتي لو النص القانوني نفسه متاح للعامة
RAG DB Questions
│ │
▼ ▼
Retrieve Ground Truth
│ │
└──────────┬──────────┘
▼
Evaluation"""

with open("/content/result_digital_pdf_test.txt", "w", encoding="utf-8") as f:
    f.write(text)

In [14]:
!ls -la /content/result_digital_pdf_test.txt
!cat /content/result_digital_pdf_test.txt

-rw-r--r-- 1 root root 21553 Sep  8 05:14 /content/result_digital_pdf_test.txt
NTI_Graduation project
OCR + : مشروع قوي فع ال ، واالهم ان كل تقنية فيه لها دور حقيقيSmart Citizen Assistant المشروع اللي اخترتيه
.Transformers + RAG + Vector DB + Hybrid Search + Router + LLM Chains + Verification
.سيناريو كامل من لحظة ما المواطن يكتب السؤال لحد ما ياخد االجابة خليني اف كهولك من الصفر، وبعدها امشي معاكي في
او ال : المشروع ببساطة بيعمل ايه؟🧠
: مواقع حكومية مختلفة3 تخيلي ان المواطن بدل ما يدخل علي
. واحد ويقول له اي حاجةChatbot يدخل علي
:مث
""عايز اطلع بطاقة رقم قومي الول مرة، ايه االوراق المطلوبة؟
:او
""عندي مخالفة مرورية، اعرف اعمل ايه؟
:/صورة ويقولPDF او يرفع
""المستند ده ناقصه ايه؟
:النظام نفسه يفهم
السؤال تابع النهي جهة؟
:ثم
هل محتاج نبحث في المستندات؟
:ثم
ايه المعلومات الموجودة في القوانين واللوائح المتعلقة بالسؤال؟
:ثم
االحوال المدنية
الضرائب
المرور
.يولد اجابة بنا ء علي المصادر الرسمية
:ثم
.يتحقق ان االجابة فع ال مدعومة بالمصادر
.Chatbot يعني مش مجرد
:هو اقرب الي
Intelligent Governmen